# label_dropout × SSR Experiment

**Question:** Does `label_dropout` during training affect the Spread/Skill Ratio (SSR) in autoregressive ensemble forecasting?

**Design:**
- Train one 2-level conditional flow-matching model per `label_dropout` value
- Early stopping (patience=5) on val loss, max 30 epochs
- Evaluate each model with autoregressive ensemble rollout (ODE steps=100)
- Metrics: RMSE, CRPS, SSR vs lead time

**Section 10** (quick, no retraining) sweeps the CFG guidance weight on a chosen
trained model to check whether underdispersion can be fixed purely at inference time.

**Architecture (2-level, matches `trainCondDiffusion.ipynb`):**
- `in_channels=4`  — 2 ch (z_s) + 2 ch (x_t conditioning)
- `out_channels=2`
- All other hyperparameters identical to `trainCondDiffusion.ipynb`

## 0. Imports

In [15]:
import sys
import copy
import csv
from pathlib import Path

import numpy as np
import torch
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
import matplotlib.pyplot as plt
from natsort import natsorted

SQG_DIR = Path('SQG')
sys.path.insert(0, str(SQG_DIR))

from diffusion_networks import SongUNet

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

Device: cuda


## 1. Config

In [ ]:
# ── data ──────────────────────────────────────────────────────────────────
DATA_DIR    = Path('../data')
DATA_STD    = 2660.0
IMG_CHANNELS   = 2        # SQG has 2 vertical levels
IMG_RESOLUTION = 64
TRAIN_FRAC  = 0.8

# ── model ─────────────────────────────────────────────────────────────────
FILTERS     = 32

# ── experiment ────────────────────────────────────────────────────────────
LABEL_DROPOUT_VALUES = [0.5, 0.3, 0.2, 0.1, 0.05, 0.0]

# ── training ──────────────────────────────────────────────────────────────
BATCH_SIZE   = 16
MAX_EPOCHS   = 30
PATIENCE     = 5       # early stopping patience
LR           = 1e-3
WEIGHT_DECAY = 1e-4

# ── evaluation ────────────────────────────────────────────────────────────
N_ENSEMBLE    = 10
ROLLOUT_STEPS = 20
ODE_STEPS     = 100
N_INIT        = 20

# ── paths ─────────────────────────────────────────────────────────────────
RESULTS_DIR = Path('../models/results/dropout_exp')
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

def model_path(dropout):
    return RESULTS_DIR / f'model_dropout_{dropout:.3f}.pth'

def log_path(dropout):
    return RESULTS_DIR / f'log_dropout_{dropout:.3f}.csv'

## 2. Dataset — 2-level consecutive pairs

Returns `(x_t, x_{t+1})` pairs with shape `(2, H, W)`, identical to `trainCondDiffusion.ipynb`.

In [17]:
class SQGPairDataset(Dataset):
    """
    Consecutive (x_t, x_{t+1}) pairs from all SQG vertical levels.
    Returns tensors of shape (2, H, W).
    """
    def __init__(self, data_dir, std, split='train', train_frac=0.8):
        data_dir = Path(data_dir)
        files = natsorted(data_dir.glob('sqg_N64_1hrly_*.npy'))
        assert len(files) > 0, f'No files in {data_dir}'

        cut   = int(len(files) * train_frac)
        files = files[:cut] if split == 'train' else files[cut:]

        self.pairs = []
        self.data  = []
        for i, f in enumerate(files):
            arr = torch.tensor(
                np.load(f).astype(np.float32) / std
            )                          # (T, 2, H, W)
            self.data.append(arr)
            T = arr.shape[0]
            for t in range(T - 1):
                self.pairs.append((i, t))

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        i, t = self.pairs[idx]
        return self.data[i][t], self.data[i][t + 1]


train_dataset = SQGPairDataset(DATA_DIR, DATA_STD, split='train')
val_dataset   = SQGPairDataset(DATA_DIR, DATA_STD, split='val')

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f'Train pairs: {len(train_dataset)},  Val pairs: {len(val_dataset)}')
x_t, x_t1 = train_dataset[0]
print(f'Sample shape: {x_t.shape}')

Train pairs: 8000,  Val pairs: 2000
Sample shape: torch.Size([2, 64, 64])


## 3. Model builder and training utilities

In [ ]:
def build_model(label_dropout):
    return SongUNet(
        img_resolution     = IMG_RESOLUTION,
        in_channels        = IMG_CHANNELS * 2,   # 2 (z_s) + 2 (x_t conditioning)
        out_channels       = IMG_CHANNELS,
        embedding_type     = 'fourier',
        encoder_type       = 'residual',
        decoder_type       = 'standard',
        channel_mult_noise = 2,
        resample_filter    = [1, 3, 3, 1],
        model_channels     = FILTERS,
        channel_mult       = [2, 2, 2],
        attn_resolutions   = [32],
        label_dropout      = label_dropout,
    ).to(device)


def flow_matching_loss(model, x_t, x_t1):
    B         = x_t1.shape[0]
    s         = torch.rand(B, device=x_t1.device)
    s_spatial = s.view(B, 1, 1, 1)
    z0        = torch.randn_like(x_t1)
    z_s       = (1 - s_spatial) * z0 + s_spatial * x_t1
    target    = x_t1 - z0
    pred      = model(z_s, s, class_labels=x_t)
    return ((pred - target) ** 2).mean()


def train_one_model(label_dropout):
    """
    Train a model with the given label_dropout.
    Returns (best_val_loss, epochs_trained, log).
    Saves best checkpoint to model_path(label_dropout).
    """
    print(f'\n{"="*60}')
    print(f'Training  label_dropout={label_dropout}')
    print(f'{"="*60}')

    model     = build_model(label_dropout)
    optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=MAX_EPOCHS)
    warmup    = optim.lr_scheduler.LinearLR(
        optimizer, start_factor=0.01, end_factor=1.0, total_iters=500
    )

    best_val   = float('inf')
    no_improve = 0
    log        = []

    with open(log_path(label_dropout), 'w', newline='') as f:
        csv.writer(f).writerow(['epoch', 'train_loss', 'val_loss'])

    for epoch in range(MAX_EPOCHS):
        # training
        model.train()
        total_train = 0.0
        for x_t, x_t1 in tqdm(train_loader, desc=f'  Epoch {epoch+1}/{MAX_EPOCHS} [train]', leave=False):
            x_t, x_t1 = x_t.to(device), x_t1.to(device)
            optimizer.zero_grad()
            loss = flow_matching_loss(model, x_t, x_t1)
            loss.backward()
            optimizer.step()
            warmup.step()
            total_train += loss.item()
        avg_train = total_train / len(train_loader)

        # validation
        model.eval()
        total_val = 0.0
        with torch.no_grad():
            for x_t, x_t1 in val_loader:
                x_t, x_t1 = x_t.to(device), x_t1.to(device)
                total_val += flow_matching_loss(model, x_t, x_t1).item()
        avg_val = total_val / len(val_loader)

        scheduler.step()
        log.append((epoch + 1, avg_train, avg_val))

        with open(log_path(label_dropout), 'a', newline='') as f:
            csv.writer(f).writerow([epoch + 1, avg_train, avg_val])

        tag = ''
        if avg_val < best_val:
            best_val   = avg_val
            no_improve = 0
            torch.save(model.state_dict(), model_path(label_dropout))
            tag = '  <- best'
        else:
            no_improve += 1

        print(f'  Epoch {epoch+1:3d}  train={avg_train:.4f}  val={avg_val:.4f}{tag}')

        if no_improve >= PATIENCE:
            print(f'  Early stop at epoch {epoch+1} (no improvement for {PATIENCE} epochs)')
            break

    print(f'  Best val loss: {best_val:.4f}  |  saved to {model_path(label_dropout)}')
    return best_val, len(log), log

## 4. Run training for all label_dropout values

Each model is saved to `models/results/dropout_exp/model_dropout_{value}.pth`.
Delete the `.pth` files first if you want to retrain from scratch.

In [ ]:
training_summary = {}   # label_dropout -> (best_val_loss, epochs_trained)

for dropout in LABEL_DROPOUT_VALUES:
    best_val, n_epochs, _ = train_one_model(dropout)
    training_summary[dropout] = (best_val, n_epochs)

print('\n--- Training summary ---')
print(f'{"dropout":>10}  {"best_val":>10}  {"epochs":>8}')
for d, (v, e) in training_summary.items():
    print(f'{d:10.3f}  {v:10.4f}  {e:8d}')

## 5. Plot training curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
colors = plt.cm.plasma(np.linspace(0.1, 0.9, len(LABEL_DROPOUT_VALUES)))

for ax, metric_idx, title in [
    (axes[0], 1, 'Train loss'),
    (axes[1], 2, 'Val loss'),
]:
    for dropout, color in zip(LABEL_DROPOUT_VALUES, colors):
        lp = log_path(dropout)
        if not lp.exists():
            continue
        data = np.loadtxt(lp, delimiter=',', skiprows=1)
        if data.ndim == 1:
            data = data[np.newaxis]
        ax.plot(data[:, 0], data[:, metric_idx], color=color, label=f'dropout={dropout}')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.set_title(title)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.suptitle('Training curves per label_dropout')
plt.tight_layout()
plt.show()

## 6. Evaluation setup

In [18]:
# load val trajectories (2-level)
files     = natsorted(DATA_DIR.glob('sqg_N64_1hrly_*.npy'))
cut       = int(len(files) * TRAIN_FRAC)
val_files = files[cut:]

val_trajs = []
for f in val_files:
    arr = torch.tensor(np.load(f).astype(np.float32) / DATA_STD)  # (T, 2, H, W)
    val_trajs.append(arr)

T_per_traj = val_trajs[0].shape[0]
max_t      = T_per_traj - ROLLOUT_STEPS - 1

rng          = np.random.default_rng(42)
traj_indices = rng.integers(0, len(val_trajs), size=N_INIT)
time_indices = rng.integers(0, max_t,          size=N_INIT)
init_points  = list(zip(traj_indices.tolist(), time_indices.tolist()))

print(f'Val trajectories: {len(val_trajs)},  T={T_per_traj},  init points: {N_INIT}')
print(f'Data shape: {val_trajs[0].shape}')

Val trajectories: 20,  T=101,  init points: 20
Data shape: torch.Size([101, 2, 64, 64])


In [22]:
@torch.no_grad()
def sample_one_step(model, x_t, steps=ODE_STEPS, guidance_weight=1.0):
    """
    One conditional flow ODE solve: noise -> x_{t+1}.

    guidance_weight=1.0 : standard conditional sampling
    guidance_weight<1.0 : less conditioning (more spread via CFG)
    guidance_weight=0.0 : fully unconditional

    For the unconditional pass, zeros are passed as class_labels (not None),
    because the model was trained with label_dropout by zeroing out the conditioning,
    so the first conv layer always expects 4 channels (z_s + cond).
    """
    B  = x_t.shape[0]
    dt = 1.0 / steps
    z  = torch.randn_like(x_t)

    for i in range(steps):
        s = torch.full((B,), i * dt, device=x_t.device)
        if guidance_weight == 1.0:
            b = model(z, s, class_labels=x_t)
        else:
            b_cond   = model(z, s, class_labels=x_t)
            b_uncond = model(z, s, class_labels=torch.zeros_like(x_t))
            b = b_uncond + guidance_weight * (b_cond - b_uncond)
        z = z + b * dt
    return z


@torch.no_grad()
def autoregressive_rollout(model, x0, rollout_steps, n_ensemble, ode_steps, guidance_weight=1.0):
    """
    Returns preds: np.ndarray, shape (rollout_steps, n_ensemble, 2, H, W)
    """
    x_cur = x0.unsqueeze(0).expand(n_ensemble, -1, -1, -1).clone().to(device)
    preds = []
    for _ in range(rollout_steps):
        x_next = sample_one_step(model, x_cur, steps=ode_steps, guidance_weight=guidance_weight)
        preds.append(x_next.cpu().numpy())
        x_cur  = x_next
    return np.stack(preds, axis=0)


def compute_rmse(preds_mean, truth):
    return np.sqrt(((preds_mean - truth) ** 2).mean(axis=(1, 2, 3)))


def compute_crps(preds, truth):
    mae = np.abs(preds - truth[:, np.newaxis]).mean(axis=1)
    n   = preds.shape[1]
    s   = np.sort(preds, axis=1)
    w   = (2 * np.arange(1, n + 1) - n - 1).reshape(1, n, 1, 1, 1)
    spread = (w * s).sum(axis=1) / (n * (n - 1))
    return (mae - spread).mean(axis=(1, 2, 3))


def compute_ssr(preds, rmse):
    spread = preds.std(axis=1).mean(axis=(1, 2, 3))
    return spread / (rmse + 1e-8)


def evaluate_model(model, guidance_weight=1.0):
    """
    Autoregressive rollout over all init points.
    Returns (mean_rmse, mean_crps, mean_ssr) each shape (rollout_steps,).
    """
    model.eval()
    all_rmse, all_crps, all_ssr = [], [], []

    for ti, t in tqdm(init_points, desc='  eval init pts', leave=False):
        x0    = val_trajs[ti][t]                                       # (2, H, W)
        truth = val_trajs[ti][t+1 : t+1+ROLLOUT_STEPS].numpy()        # (steps, 2, H, W)

        preds      = autoregressive_rollout(
            model, x0, ROLLOUT_STEPS, N_ENSEMBLE, ODE_STEPS, guidance_weight
        )                                                              # (steps, n_ens, 2, H, W)
        preds_mean = preds.mean(axis=1)                                # (steps, 2, H, W)

        all_rmse.append(compute_rmse(preds_mean, truth))
        all_crps.append(compute_crps(preds, truth))
        all_ssr.append(compute_ssr(preds, all_rmse[-1]))

    return (
        np.stack(all_rmse).mean(axis=0),
        np.stack(all_crps).mean(axis=0),
        np.stack(all_ssr).mean(axis=0),
    )

## 7. Evaluate all trained models

In [ ]:
results = {}   # label_dropout -> {'rmse': ..., 'crps': ..., 'ssr': ...}

for dropout in LABEL_DROPOUT_VALUES:
    mp = model_path(dropout)
    if not mp.exists():
        print(f'  SKIP dropout={dropout}: checkpoint not found at {mp}')
        continue

    print(f'\nEvaluating dropout={dropout} ...')
    m = build_model(dropout)
    m.load_state_dict(torch.load(mp, map_location=device, weights_only=True))

    rmse, crps, ssr = evaluate_model(m)
    results[dropout] = {'rmse': rmse, 'crps': crps, 'ssr': ssr}

    print(f'  Lead-1  RMSE={rmse[0]:.4f}  CRPS={crps[0]:.4f}  SSR={ssr[0]:.4f}')
    print(f'  Lead-10 RMSE={rmse[9]:.4f}  CRPS={crps[9]:.4f}  SSR={ssr[9]:.4f}')
    print(f'  Lead-20 RMSE={rmse[19]:.4f}  CRPS={crps[19]:.4f}  SSR={ssr[19]:.4f}')

## 8. Comparison plots — metrics vs lead time

In [ ]:
lead_times = np.arange(1, ROLLOUT_STEPS + 1)
colors     = plt.cm.plasma(np.linspace(0.1, 0.9, len(LABEL_DROPOUT_VALUES)))

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for dropout, color in zip(LABEL_DROPOUT_VALUES, colors):
    if dropout not in results:
        continue
    r = results[dropout]
    label = f'dropout={dropout}'
    axes[0].plot(lead_times, r['rmse'], color=color, marker='o', ms=3, label=label)
    axes[1].plot(lead_times, r['crps'], color=color, marker='o', ms=3, label=label)
    axes[2].plot(lead_times, r['ssr'],  color=color, marker='o', ms=3, label=label)

axes[2].axhline(1.0, color='red', linestyle='--', label='ideal SSR=1')

for ax, title, ylabel in [
    (axes[0], 'RMSE (lower is better)', 'RMSE'),
    (axes[1], 'CRPS (lower is better)', 'CRPS'),
    (axes[2], 'SSR (ideal = 1)',        'SSR'),
]:
    ax.set_title(title)
    ax.set_xlabel('Lead time (steps)')
    ax.set_ylabel(ylabel)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.suptitle(f'Autoregressive evaluation — 2 levels  |  ensemble={N_ENSEMBLE}', y=1.02)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'metrics_vs_lead.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Summary — SSR / RMSE / CRPS at fixed lead times vs label_dropout

In [ ]:
probe_leads    = [l for l in [1, 5, 10, 20] if l <= ROLLOUT_STEPS]
dropouts_done  = [d for d in LABEL_DROPOUT_VALUES if d in results]
markers        = ['o', 's', '^', 'D']

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, key, title in [
    (axes[0], 'rmse', 'RMSE vs label_dropout'),
    (axes[1], 'crps', 'CRPS vs label_dropout'),
    (axes[2], 'ssr',  'SSR vs label_dropout'),
]:
    for lead, marker in zip(probe_leads, markers):
        vals = [results[d][key][lead - 1] for d in dropouts_done]
        ax.plot(dropouts_done, vals, marker=marker, label=f'lead={lead}')
    if key == 'ssr':
        ax.axhline(1.0, color='red', linestyle='--', label='ideal=1')
    ax.set_xlabel('label_dropout')
    ax.set_ylabel(key.upper())
    ax.set_title(title)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.suptitle('Effect of label_dropout on forecast quality', y=1.02)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'metrics_vs_dropout.png', dpi=150, bbox_inches='tight')
plt.show()

## 10. CFG guidance sweep — inference-time diagnostic

For models with `label_dropout > 0`, CFG lets you trade off spread vs skill at inference
**without retraining** by interpolating conditional and unconditional velocity:

> `v = v_uncond + w * (v_cond - v_uncond)`
> - `w=1` : standard conditional (current baseline)
> - `w<1` : less conditioning → more spread → higher SSR but higher RMSE
> - `w=0` : fully unconditional

Run this to check whether underdispersion is fixable purely at inference time.

In [23]:
# Change CFG_DROPOUT to whichever model performed best in Section 9.
CFG_DROPOUT = 0.1
CFG_WEIGHTS = [0.0, 0.3, 0.5, 0.7, 1.0]

# mp = model_path(CFG_DROPOUT)
# assert mp.exists(), f'No checkpoint for dropout={CFG_DROPOUT}'
mp = "../models/results/best_model_conditional.pth"

cfg_model = build_model(CFG_DROPOUT)
cfg_model.load_state_dict(torch.load(mp, map_location=device, weights_only=True))

cfg_results = {}
for w in CFG_WEIGHTS:
    print(f'CFG weight w={w} ...')
    rmse, crps, ssr = evaluate_model(cfg_model, guidance_weight=w)
    cfg_results[w] = {'rmse': rmse, 'crps': crps, 'ssr': ssr}
    print(f'  Lead-1  SSR={ssr[0]:.4f}  RMSE={rmse[0]:.4f}')
    print(f'  Lead-10 SSR={ssr[9]:.4f}  RMSE={rmse[9]:.4f}')

CFG weight w=0.0 ...


  Lead-1  SSR=0.8987  RMSE=1.0119
  Lead-10 SSR=0.8788  RMSE=1.0249
CFG weight w=0.3 ...


  Lead-1  SSR=0.8493  RMSE=0.4060
  Lead-10 SSR=0.4901  RMSE=0.9668
CFG weight w=0.5 ...


  Lead-1  SSR=0.9041  RMSE=0.2066
  Lead-10 SSR=0.3968  RMSE=0.9025
CFG weight w=0.7 ...


  Lead-1  SSR=1.0498  RMSE=0.0888
  Lead-10 SSR=0.2373  RMSE=0.5513
CFG weight w=1.0 ...


  Lead-1  SSR=0.8392  RMSE=0.0193
  Lead-10 SSR=0.4910  RMSE=0.0929


In [1]:
colors_cfg = plt.cm.viridis(np.linspace(0.1, 0.9, len(CFG_WEIGHTS)))

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for w, color in zip(CFG_WEIGHTS, colors_cfg):
    r = cfg_results[w]
    axes[0].plot(lead_times, r['rmse'], color=color, marker='o', ms=3, label=f'w={w}')
    axes[1].plot(lead_times, r['crps'], color=color, marker='o', ms=3, label=f'w={w}')
    axes[2].plot(lead_times, r['ssr'],  color=color, marker='o', ms=3, label=f'w={w}')

axes[2].axhline(1.0, color='red', linestyle='--', label='ideal SSR=1')

for ax, title, ylabel in [
    (axes[0], 'RMSE',  'RMSE'),
    (axes[1], 'CRPS',  'CRPS'),
    (axes[2], 'SSR',   'SSR'),
]:
    ax.set_title(f'{title} — CFG sweep (dropout={CFG_DROPOUT})')
    ax.set_xlabel('Lead time (steps)')
    ax.set_ylabel(ylabel)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.suptitle(f'CFG guidance weight sweep — label_dropout={CFG_DROPOUT}', y=1.02)
plt.tight_layout()
plt.savefig(RESULTS_DIR / f'cfg_sweep_dropout{CFG_DROPOUT}.png', dpi=150, bbox_inches='tight')
plt.show()

NameError: name 'plt' is not defined